# Im ersten Teil ist dargestellt, wie OCR in dieser Datei betrieben wurde. Im zweiten Teil ist dargestellt, wie entstandene Fehler behoben wurden

In [ ]:
import pandas as pd
import numpy as np
import easyocr
import os
import tqdm
import tqdm.notebook
from tqdm.notebook import tqdm
import re


pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True


os.chdir(r"C:\Users\hundh\Desktop")

## Teil 1

In [ ]:
# Define the path to the images folder
images_root_path = 'Python/Bilder'

# Initialize the EasyOCR reader
reader = easyocr.Reader(['de'], gpu=True)

# Initialize a dictionary to store OCR results
ocr_results = {}

# Loop through each subfolder in the images folder
for root, dirs, files in os.walk(images_root_path):
    for file in tqdm(files, desc=f"Processing images in {root}"):
        if file.endswith(('.jpg', '.jpeg', '.png')):  # Add more image file extensions if needed
            image_path = os.path.join(root, file)
            author = os.path.basename(root)
            image_id, _ = os.path.splitext(file)

            # Read the image using EasyOCR
            text = reader.readtext(image_path)

            # Extracted text as a single string
            extracted_text = ' '.join([line[1] for line in text])

            # Store the result in the dictionary
            ocr_results[(author, image_id)] = extracted_text

In [ ]:
ocr_results

In [ ]:
# Aus ocr-dictionary wird dataframe
ocr = pd.DataFrame(ocr_results.items(), columns=["author_filenameimage", "ocr_text"])

In [ ]:
#In einer Variable sind sowohl author als auch filename, deshalb werden sie getrennt
ocr["author_filenameimage"] = ocr["author_filenameimage"].astype(str)
ocr[["author","filename_image"]] = ocr["author_filenameimage"].str.split(", ", expand=True)

In [ ]:
ocr

In [ ]:
# Bereinigung
ocr["author"] = ocr["author"].str.replace("('","")
ocr["author"] = ocr["author"].str.replace("'","")
ocr["filename_image"] = ocr["filename_image"].str.replace("'","")
ocr["filename_image"] = ocr["filename_image"].str.replace(")","")
ocr[["id_scraping","nutzlos_1","nutzlos_2"]] = ocr["author"].str.split("_", expand=True)
ocr["id_scraping"] = ocr["id_scraping"].astype(int)
ocr = ocr.drop(["nutzlos_1","nutzlos_2","author_filenameimage","author"], axis=1)

## Teil 2: Fehlerbehebung. Dieser Teil entstand erst nachdem alle Posts durch OCR-liefen

Beim Vorgehen mit OCR sind zwei Fehler entstanden:
1. OCR wurde auf Google colab und mittels einer Laptop-CPU (in diesem OCR-Code mit dem obigen Vorgehen) betrieben. Beim dreifachen mergen (ocr-Texte von Colab + ocr-Texte mit Laptop-CPU + originaler Datensatz) wurden Reihen unbeabsichtigt kopiert, wodurch im Datensatz mit OCR-Beiträgen mehr Reihen sind als im originalen Datensatz. Die ocr-Texte, die mittels Laptop CPU-gewonnen wurden sind nur noch in der durch die kopierten Reihen verfälschten Datei (hier: df_voll/ data_btw_final_mit_ocr) enthalten.
2. Die OCR-Beiträge wurden durch das wiederholte reinladen zum Teil "aneinandergeheftet". Es wiederholen sich Texte also manchmal innerhalb einer Spalte (fiktives Beispiel: "Ich bin fan der CDUIch bin fan der CDU").  

Diese Fehler werden hier behoben

Schritt 1: Die OCR-Texte (die hier extrahiert wurden), die nur noch im falschen Datensatz enthalten sind, werden aus diesem extrahiert und ein neuer, richtiger Datensatz erstellt 

In [ ]:
# df_voll ist der Datensatz mit allen OCR-Texten + fälschlicherweise kopierten Reihen. Das ist also der falsche Datensatz
df_voll = pd.read_csv("Python/Daten/data_btw_final_ocr_FALSCH.csv")
print(df_voll.shape[0])
# Das ist der richtige Datensatz ohne ocr-Texte
df_leer = pd.read_csv("Python/Daten/data_btw_final.csv") 
print(df_leer.shape[0])

In [ ]:
# Hier werden die ocr-Texte von Colab geladen und doppelte Zeilen gelöscht (und Müller ausgeschlossen)

ocr_colab = pd.read_csv("Python/Daten/ocr_colab.csv")
ocr_colab.shape[0]
ocr_colab = ocr_colab.drop_duplicates(subset=["filename_image","id_scraping", "ocr_text"]) #Einfach doppelte Zeilen (z.B. wenn bei Hälfte der Bilder für einen Kandidierenden abgebrochen und neu geladen wurde)
ocr_colab = ocr_colab[ocr_colab["id_scraping"] != 3176]

In [ ]:
# Der falsche Datensatz wird mit ocr_colab gemerged. Durch "indicator=True" ist erkennbar, ob die ocr-texte im falschen Datensatz aus der ocr-colab datei stammen. 
# Die OCR-Texte aus dem OCR-Dokument sind nur in df_voll und nicht in ocr_colab.

neu = pd.merge(df_voll, ocr_colab, how="outer", on=["filename_image","id_scraping"], indicator=True)
neu = neu.drop_duplicates()
neu.shape[0]

In [ ]:
# Bei "_merge== left_only" können die Texte nur aus dem OCR-Dokument und nicht aus ocr_colab stammen. Diese werden in einem neuen Datensatz gesammelt

main_ocr = neu[neu["_merge"] == "left_only"]
del main_ocr["ocr_text_y"]
main_ocr.rename(columns={"ocr_text_x":"ocr_text"}, inplace=True)

In [ ]:
# Dadurch haben main_ocr und ocr_colab die gleiche Struktur
main_ocr = main_ocr[["ocr_text","filename_image","id_scraping"]]
print(main_ocr.shape[0])
print(ocr_colab.shape[0])

In [ ]:
# In ocr_weiter sind jetzt sowohl die ocr-Texte von colab als auch die ocr_texte aus dem OCR-Dokument enthalten
ocr_weiter = pd.concat([main_ocr, ocr_colab])

In [ ]:
# Um mit filename mergen zu können, müssen im originalen Datensatz erst die Zusätze hinter den filenames gelöscht werden

df_leer["filename_image"] = df_leer["filename_image"].str.replace(".jpg","")
df_leer["filename_image"] = df_leer["filename_image"].str.replace(".png","")
df_leer["filename_image"] = df_leer["filename_image"].str.replace(".jpeg","")
df_leer["filename_image"] = df_leer["filename_image"].str.replace(".webp","")

In [ ]:
# Jetzt wird der originale Datensatz ohne ocr-Texte (df_leer) mit den OCR-Texten mittels scraping id und filename_image gemerged
df_mit_ocr = pd.merge(df_leer, ocr_weiter,how="outer", on=["filename_image","id_scraping"])

In [ ]:
# Beim mergen entstehen erneut ein paar unnötige Zeilen (wohl weil "filename_image" + "id_scraping" nicht immer unique ist). Diese sind daran erkenntlich, dass sie keine id haben oder komplette Duplikate anderer Zeilen sind.
# Zudem gibt es ein paar Bilder, die doppelt durch "geocrd" wurden, wenn der Prozess beim Durchlaufen durch einen Kandidierenden-Ordner abgebrochen und später nochmal neu angefangen wurde

df_mit_ocr.dropna(subset="id",inplace=True)
df_mit_ocr.drop_duplicates(subset=["url","post_source_domain","author","body","filename_image","acc_type","acc_name"], inplace=True)
print(df_leer.shape[0])
print(df_mit_ocr.shape[0]) # Der neue Datensatz mit ocr-Texten enthält jetzt genau so viele Reihen wie der originale Datensatz ohne ocr-Texte

In [ ]:
# Um zu checken, ob die gleiche Reihenanzahl von df_leer und df_mit_ocr nicht nur zufällig ist, wird gemerged und über den Indikator nachgesehen, ob alle Reihen in beiden Datensätzen vorhanden sind
test = pd.merge(df_leer, df_mit_ocr, indicator=True) 

# Wenn in beiden Datensätzen alle Reihen gleich sind, ist _merge=both. Das ist hier der fall
print(test.shape[0])
test.groupby(by="_merge")["_merge"].size()

Schritt 2: Die aneinandergereihten Sätze in ocr_text werden identifiziert und die Wiederholungen entfernt (der originale Text bleibt bestehen) 

In [ ]:
# Der Repeater findet sich wiederholende Texte. In der Spalte "pattern" ist zu sehen, was der originale Text sein muss (also der sich nicht wiederholende Text)
# Die Variable ocr_text wird überschrieben: Wenn es ein pattern gibt, wird der originale (sich nicht wiederholende) Text wiederhergestellt, wenn nicht, wird der vorhandene Text behalten
# Theoretisch (aber unwahrscheinlich) könnten dadurch echte Posts verfälscht werden, wenn sich ihr Inhalt doppelt. Für die Klassifizierung sollte das aber kein Problem sein, da ja der Inhalt trotzdem noch einmal vorkommt (Ob in einem Post "Ich liebe Klimaschutz Ich liebe Klimaschutz" oder nur "Ich liebe Klimaschutz" steht, wird sich wohl nicht auf die Klassifizierung auswirken) 

df_mit_ocr["ocr_text"] = df_mit_ocr["ocr_text"].astype(str)

REPEATER = re.compile(r"(.+?)\1+$")

def repeated(s):
    match = REPEATER.match(s)
    return match.group(1) if match else None

df_mit_ocr["pattern"] = df_mit_ocr["ocr_text"].apply(repeated)

df_mit_ocr["ocr_text"] = df_mit_ocr.apply(
    lambda row: row["pattern"] if row["pattern"] else row["ocr_text"], axis=1)

In [ ]:
df_mit_ocr.to_csv("Python/Daten/data_btw_final_ocr.csv")